# GRPO for Math Reasoning on a Free Colab T4

Reinforcement learning post-training of Qwen2.5-1.5B with verifiable rewards, on the free tier.

**Runtime > Change runtime type > T4 GPU** before running.

Sections 1 to 3 are independent, so you can run any one of them on its own.
Section 1 is the headline result (~40 min). Section 2 measures the seed noise floor, which is what
makes every other comparison readable. Section 3 is the full reproduction across several sessions.

## Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git clone -q https://github.com/USER/grpo-math.git 2>/dev/null || true
%cd grpo-math
!pip install -q -r requirements.txt

## 1. GSM8K: train and measure the gain

Baseline first, so the comparison is on the same eval in the same session.
Rank 64 is the best setting found in the sweep, so this is the headline configuration.

In [ ]:
!python evaluate.py --model Qwen/Qwen2.5-1.5B-Instruct --tag base \
    --dataset gsm8k --n 8 --limit 500 --ks 1,4,8 --max-tokens 640 \
    --gpu-mem 0.70 --out results/gsm8k.jsonl

In [ ]:
!python train.py --name lora_r64 --dataset gsm8k --steps 100 \
    --num-generations 8 --batch-size 16 --grad-accum 2 --max-tokens 640 \
    --lr 1e-5 --lora-r 64 --vllm-mem 0.25

In [ ]:
!python evaluate.py --model runs/lora_r64/final --tag lora_r64 \
    --dataset gsm8k --n 8 --limit 500 --ks 1,4,8 --max-tokens 640 \
    --gpu-mem 0.70 --out results/sweep.jsonl

Compare `avg@1` against `pass@8` in the two results above.

If `avg@1` rose a lot but `pass@8` barely moved, RL concentrated probability onto answers the base
model could already reach rather than teaching it anything new. That is what happens here, and it is
the single most useful thing to understand about RLVR.

## 2. The seed noise floor

Three runs identical except for the seed. Whatever spread you see here is the smallest difference
anywhere else in this notebook that is worth believing. Skipping this step is how people convince
themselves a 1-point difference is a finding.

In [ ]:
for s in range(3):
    !python train.py --name grpo_s{s} --dataset gsm8k --steps 100 \
        --num-generations 8 --batch-size 16 --grad-accum 2 --max-tokens 640 \
        --loss-type grpo --lr 1e-5 --lora-r 16 --seed {s} --vllm-mem 0.25
    !python evaluate.py --model runs/grpo_s{s}/final --tag grpo_s{s} \
        --dataset gsm8k --n 8 --limit 500 --ks 1,4,8 --max-tokens 640 \
        --gpu-mem 0.70 --out results/seeds.jsonl

## 3. A negative result: multiplication by curriculum

Train on 1-to-4 digit multiplication only, then test on held-out 5- and 6-digit problems. If RL
taught an algorithm rather than a lookup table, this should extrapolate.

It mostly does not, and it is unstable across seeds: one of the three seeds in the README collapsed
below the base model. Run at least two seeds before believing whatever you get.

In [ ]:
!python evaluate.py --model Qwen/Qwen2.5-1.5B-Instruct --dataset mult5 --tag base_d5 \
    --n 32 --limit 512 --ks 1,8,32 --max-tokens 1024 --gpu-mem 0.70 --out results/mult.jsonl

In [ ]:
!python train.py --name mult_s0 --dataset mult_curriculum --steps 100 \
    --num-generations 8 --batch-size 16 --grad-accum 2 --max-tokens 1024 \
    --lr 1e-4 --lora-r 16 --vllm-mem 0.25 --seed 0

In [ ]:
!python evaluate.py --model runs/mult_s0/final --dataset mult5 --tag mult_s0_d5 \
    --n 32 --limit 512 --ks 1,8,32 --max-tokens 1024 --gpu-mem 0.70 --out results/mult.jsonl

## 4. Full reproduction

Every number in the README. Roughly 10 to 15 GPU-hours on a T4, so expect two or three Colab
sessions. `run_all.sh` skips any run or evaluation that already finished, so just rerun it: keep
`runs/` and `results/` between sessions by saving them to Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/grpo-math && ln -sfn /content/drive/MyDrive/grpo-math/runs runs

In [ ]:
!bash run_all.sh

## Figures and tables

In [ ]:
!python plots.py && python report.py
from IPython.display import Image, display
import glob
for p in sorted(glob.glob('figures/*.png')):
    display(Image(p))